In [ ]:
# Setup
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Scanpy settings
sc.settings.verbosity = 3  # Verbose output
sc.settings.set_figure_params(dpi=150, facecolor='white', figsize=(8, 6))

# Paths
from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_raw = os.path.join(base_path, "Data/raw")
data_processed = os.path.join(base_path, "Data/processed")
results_path = os.path.join(base_path, "results")

print(" Imports successful!")
print(f"Scanpy version: {sc.__version__}")
print(f"\nPaths configured:")
print(f"  Raw data: {data_raw}")
print(f"  Processed data: {data_processed}")
print(f"  Results: {results_path}")

In [ ]:
# Load Walesky primary dataset
walesky_file = os.path.join(data_raw, "walesky_GSE136679/GSE136679_ALI_filterHQhep.txt")

print(f"Loading file: {walesky_file}")
print(f"File exists: {os.path.exists(walesky_file)}\n")

# Load data
# Format: Tab-separated text file, genes as rows, cells as columns
# We transpose (.T) to get cells as rows (scanpy convention)
adata_walesky = sc.read_text(walesky_file, delimiter='\t').T

print(" Walesky dataset loaded successfully!\n")
print(f" Dataset dimensions:")
print(f"   Cells (observations): {adata_walesky.n_obs:,}")
print(f"   Genes (variables): {adata_walesky.n_vars:,}")
print(f"   Shape: {adata_walesky.shape}")

In [ ]:
# Inspect cell and gene names
print(" First 10 cell IDs (barcodes):")
print(adata_walesky.obs_names[:10].tolist())

print("\n First 10 gene names:")
print(adata_walesky.var_names[:10].tolist())

print("\n Last 10 cell IDs:")
print(adata_walesky.obs_names[-10:].tolist())

In [ ]:
# Check data structure
print(" AnnData object structure:")
print(adata_walesky)

print("\n Observations (cells) metadata:")
print(adata_walesky.obs.head())

print("\n Variables (genes) metadata:")
print(adata_walesky.var.head())

print("\n Data matrix (first 5 cells, first 5 genes):")
print(adata_walesky.X[:5, :5])

In [ ]:
# Extract metadata from cell IDs

# Extract sample code from cell IDs (everything after the underscore)
adata_walesky.obs['sample'] = adata_walesky.obs_names.str.split('_').str[1]

# Show unique samples
print(" Unique samples found:")
print(adata_walesky.obs['sample'].value_counts().sort_index())

In [ ]:
# Extract metadata from cell IDs 

# Extract sample code from cell IDs
adata_walesky.obs['sample'] = adata_walesky.obs_names.str.split('_').str[1]

# Create treatment labels
def get_treatment(sample_code):
    """Extract treatment from sample code"""
    if sample_code.startswith('UT'):
        return 'Control'
    elif sample_code.startswith('A'):
        return 'APAP'
    elif sample_code.startswith('PH'):
        return 'PartialHepatectomy'
    else:
        return 'Unknown'

adata_walesky.obs['treatment'] = adata_walesky.obs['sample'].apply(get_treatment)

# Create timepoint labels
def get_timepoint(sample_code):
    """Extract timepoint from sample code"""
    if sample_code.startswith('UT'):
        return '0h'
    elif sample_code.startswith('A6'):
        return '6h'
    elif sample_code.startswith('A24'):
        return '24h'
    elif sample_code.startswith('A48'):
        return '48h'
    elif sample_code.startswith('A96'):
        return '96h'
    elif sample_code.startswith('PH3'):
        return '3h'
    elif sample_code.startswith('PH48'):
        return '48h'
    elif sample_code.startswith('PH120'):
        return '120h'
    else:
        return 'Unknown'

adata_walesky.obs['timepoint'] = adata_walesky.obs['sample'].apply(get_timepoint)

# Extract mouse replicate number
adata_walesky.obs['mouse'] = adata_walesky.obs['sample'].str.extract(r'M(\d+)')[0]

# Create numeric timepoint for trajectory analysis
timepoint_to_hours = {
    '0h': 0, '3h': 3, '6h': 6, '24h': 24, '48h': 48, '96h': 96, '120h': 120
}
adata_walesky.obs['timepoint_hours'] = adata_walesky.obs['timepoint'].map(timepoint_to_hours)

# Summary statistics
print(" DATASET SUMMARY:")
print(f"Total cells: {adata_walesky.n_obs:,}")
print(f"Total genes: {adata_walesky.n_vars:,}\n")

print(" Treatment distribution:")
print(adata_walesky.obs['treatment'].value_counts())

print("\n Timepoint distribution:")
print(adata_walesky.obs['timepoint'].value_counts().sort_index())

print("\n Mouse replicates per condition:")
print(adata_walesky.obs.groupby(['treatment', 'timepoint'])['mouse'].nunique())

print("\n Cells per condition:")
condition_counts = adata_walesky.obs.groupby(['treatment', 'timepoint']).size()
print(condition_counts)

print("\n Metadata extraction complete!")

In [ ]:
# Save processed Walesky dataset

# Create output directory if it doesn't exist
os.makedirs(data_processed, exist_ok=True)

# Save the annotated dataset
output_file = os.path.join(data_processed, "walesky_raw_annotated.h5ad")
adata_walesky.write(output_file)

print(f" Dataset saved to: {output_file}")
print(f"   File size: {os.path.getsize(output_file) / 1e6:.1f} MB")
print(f"\n Saved dataset contains:")
print(f"   - {adata_walesky.n_obs:,} cells")
print(f"   - {adata_walesky.n_vars:,} genes")
print(f"   - Metadata: treatment, timepoint, sample, mouse, timepoint_hours")

In [ ]:
# Create comprehensive dataset summary

summary_data = []

for treatment in ['Control', 'APAP', 'PartialHepatectomy']:
    treatment_data = adata_walesky.obs[adata_walesky.obs['treatment'] == treatment]
    
    for timepoint in sorted(treatment_data['timepoint'].unique()):
        timepoint_data = treatment_data[treatment_data['timepoint'] == timepoint]
        
        n_cells = len(timepoint_data)
        n_mice = timepoint_data['mouse'].nunique()
        samples = timepoint_data['sample'].unique()
        
        summary_data.append({
            'Treatment': treatment,
            'Timepoint': timepoint,
            'Cells': n_cells,
            'Mice': n_mice,
            'Samples': ', '.join(sorted(samples))
        })

summary_df = pd.DataFrame(summary_data)

print(" WALESKY DATASET SUMMARY TABLE:")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

# Save summary table
summary_file = os.path.join(data_processed, "walesky_summary.csv")
summary_df.to_csv(summary_file, index=False)
print(f"\n Summary saved to: {summary_file}")

In [ ]:
# Visualize sample distribution

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Cells per treatment
treatment_counts = adata_walesky.obs['treatment'].value_counts()
axes[0].bar(treatment_counts.index, treatment_counts.values, color=['#2ecc71', '#e74c3c', '#3498db'])
axes[0].set_xlabel('Treatment', fontsize=12)
axes[0].set_ylabel('Number of Cells', fontsize=12)
axes[0].set_title('Cells per Treatment', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Cells per timepoint
timepoint_order = ['0h', '3h', '6h', '24h', '48h', '96h', '120h']
timepoint_counts = adata_walesky.obs['timepoint'].value_counts().reindex(timepoint_order, fill_value=0)
axes[1].bar(timepoint_counts.index, timepoint_counts.values, color='#9b59b6')
axes[1].set_xlabel('Timepoint', fontsize=12)
axes[1].set_ylabel('Number of Cells', fontsize=12)
axes[1].set_title('Cells per Timepoint', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

# Plot 3: Heatmap of Treatment × Timepoint
pivot_table = adata_walesky.obs.pivot_table(
    index='treatment', 
    columns='timepoint', 
    values='sample', 
    aggfunc='count',
    fill_value=0
)
pivot_table = pivot_table[timepoint_order]  # Reorder columns

sns.heatmap(pivot_table, annot=True, fmt='g', cmap='YlOrRd', ax=axes[2], cbar_kws={'label': 'Cells'})
axes[2].set_xlabel('Timepoint', fontsize=12)
axes[2].set_ylabel('Treatment', fontsize=12)
axes[2].set_title('Treatment × Timepoint Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(results_path, 'figures/walesky_sample_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()

print(" Figure saved to results/figures/walesky_sample_distribution.png")


In [ ]:
# Filter to APAP + Control only (exclude PH)

# Create filtered dataset
adata_walesky_apap = adata_walesky[
    adata_walesky.obs['treatment'].isin(['Control', 'APAP'])
].copy()

print(" FILTERED DATASET (APAP + Control only):")
print(f"Total cells: {adata_walesky_apap.n_obs:,}")
print(f"Total genes: {adata_walesky_apap.n_vars:,}\n")

print(" Treatment distribution:")
print(adata_walesky_apap.obs['treatment'].value_counts())

print("\n Timepoint distribution:")
print(adata_walesky_apap.obs['timepoint'].value_counts().sort_index())

print("\n Cells per condition:")
condition_counts = adata_walesky_apap.obs.groupby(['treatment', 'timepoint']).size()
print(condition_counts)

print("\n APAP trajectory:")
apap_only = adata_walesky_apap[adata_walesky_apap.obs['treatment'] == 'APAP']
print(apap_only.obs['timepoint'].value_counts().sort_index())

# Save filtered dataset
output_file = os.path.join(data_processed, "walesky_apap_control_raw.h5ad")
adata_walesky_apap.write(output_file)

print(f"\n Filtered dataset saved to: {output_file}")
print(f"   File size: {os.path.getsize(output_file) / 1e6:.1f} MB")

In [ ]:
# Create comprehensive dataset summary (APAP + Control only)

summary_data = []

for treatment in ['Control', 'APAP']:  # Removed PartialHepatectomy
    treatment_data = adata_walesky_apap.obs[adata_walesky_apap.obs['treatment'] == treatment]
    
    for timepoint in sorted(treatment_data['timepoint'].unique()):
        timepoint_data = treatment_data[treatment_data['timepoint'] == timepoint]
        
        n_cells = len(timepoint_data)
        n_mice = timepoint_data['mouse'].nunique()
        samples = timepoint_data['sample'].unique()
        
        summary_data.append({
            'Treatment': treatment,
            'Timepoint': timepoint,
            'Cells': n_cells,
            'Mice': n_mice,
            'Samples': ', '.join(sorted(samples))
        })

summary_df = pd.DataFrame(summary_data)

print(" WALESKY DATASET SUMMARY TABLE (APAP + Control):")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

# Save summary table
summary_file = os.path.join(data_processed, "walesky_apap_control_summary.csv")
summary_df.to_csv(summary_file, index=False)
print(f"\n Summary saved to: {summary_file}")

In [ ]:
# Visualize sample distribution (APAP + Control only)

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Cells per treatment
treatment_counts = adata_walesky_apap.obs['treatment'].value_counts()
colors = ['#2ecc71', '#e74c3c']  # Green for Control, Red for APAP
axes[0].bar(treatment_counts.index, treatment_counts.values, color=colors)
axes[0].set_xlabel('Treatment', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Cells', fontsize=12, fontweight='bold')
axes[0].set_title('Cells per Treatment', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)
# Add cell count labels on bars
for i, (idx, val) in enumerate(treatment_counts.items()):
    axes[0].text(i, val + 100, f'{val:,}', ha='center', va='bottom', fontweight='bold')

# Plot 2: Cells per timepoint
timepoint_order = ['0h', '6h', '24h', '48h', '96h']  # Removed 3h, 120h (PH timepoints)
timepoint_counts = adata_walesky_apap.obs['timepoint'].value_counts().reindex(timepoint_order, fill_value=0)
axes[1].bar(timepoint_counts.index, timepoint_counts.values, color='#9b59b6')
axes[1].set_xlabel('Timepoint', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Cells', fontsize=12, fontweight='bold')
axes[1].set_title('Cells per Timepoint', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)
# Add cell count labels
for i, val in enumerate(timepoint_counts.values):
    axes[1].text(i, val + 100, f'{val:,}', ha='center', va='bottom', fontweight='bold')

# Plot 3: Heatmap of Treatment × Timepoint
pivot_table = adata_walesky_apap.obs.pivot_table(
    index='treatment', 
    columns='timepoint', 
    values='sample', 
    aggfunc='count',
    fill_value=0
)
pivot_table = pivot_table[timepoint_order]  # Reorder columns

sns.heatmap(pivot_table, annot=True, fmt='g', cmap='YlOrRd', ax=axes[2], 
            cbar_kws={'label': 'Number of Cells'}, linewidths=0.5, linecolor='white')
axes[2].set_xlabel('Timepoint', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Treatment', fontsize=12, fontweight='bold')
axes[2].set_title('Treatment × Timepoint Matrix', fontsize=14, fontweight='bold')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()

# Create figures directory if it doesn't exist
os.makedirs(os.path.join(results_path, 'figures'), exist_ok=True)

# Save figure
fig_path = os.path.join(results_path, 'figures/walesky_apap_control_distribution.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(f" Figure saved to: {fig_path}")